In [1]:
import geonamescache

gc = geonamescache.GeonamesCache(min_city_population=500)
cities = tuple(gc.get_cities().values())

def get_city_coordinates(
    city_name,
    country_code=None):
    """
    Return (latitude, longitude) using an offline packaged database.

    When several places match, the most populous one is returned.
    """
    target_name = city_name.strip().casefold()
    target_country = country_code.strip().upper() if country_code else None

    matches = []

    for city in cities:
        names = {
            city["name"].casefold(),
            city.get("ascii", "").casefold(),
        }

        name_matches = any(
            target_name == name or target_name in name
            for name in names
            if name
        )

        country_matches = (
            target_country is None
            or city["countrycode"] == target_country
        )

        if name_matches and country_matches:
            matches.append(city)

    if not matches:
        return None

    best_match = max(
        matches,
        key=lambda city: city.get("population", 0),
    )

    return (
        float(best_match["latitude"]),
        float(best_match["longitude"]),
    )


print(get_city_coordinates("Frankfurt"))


ModuleNotFoundError: No module named 'geonamescache'

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import collections
import xarray as xr

In [25]:
tab = pd.read_parquet('deutsche-bahn-data/monthly_processed_data/data-2026-01.parquet')


In [26]:
city_selection = {city:count for city,count in collections.Counter(tab.station_name).items() if count >= 30000 and 'Hbf' in city}

In [30]:
city_locations = {}
for city in city_selection.keys():
    city_locations[city] = get_city_coordinates(city.replace(' Hbf','').replace(' (Main)',''))

In [31]:
city_locations

{'Hamburg Hbf': (53.55073, 9.99302),
 'Nürnberg Hbf': (49.45421, 11.07752),
 'Düsseldorf Hbf': (51.22172, 6.77616),
 'Hannover Hbf': (52.37052, 9.73322),
 'Köln Hbf': (50.93333, 6.95),
 'München Hbf': (48.06489, 11.66327),
 'Frankfurt (Main) Hbf': (50.11552, 8.68417),
 'Dortmund Hbf': (51.51494, 7.466),
 'Stuttgart Hbf': (48.78232, 9.17702),
 'Essen Hbf': (51.45657, 7.01228)}

In [35]:
with xr.open_dataset('../0_data/eobs/tg_ens_mean_0.25deg_reg_v33.0e.nc') as nc:
    l = []
    for city,location in city_locations.items():
        l.append(nc['tg'].sel(dict(latitude=location[0], longitude=location[1]), method='nearest'))

In [38]:
city_data = xr.concat(l, dim="city").assign_coords(city=list(city_locations.keys()))

In [39]:
city_data

<xarray.DataArray 'tg' (city: 10, time: 27759)>
array([[-0.59      ,  4.1       ,  4.37      , ...,  4.29      ,
        -1.16      ,  3.79      ],
       [-4.38      , -0.65999997,  1.52      , ..., -4.41      ,
        -1.92      , -4.27      ],
       [ 0.01      ,  3.9399998 ,  5.16      , ..., -0.41      ,
         0.07      ,  0.47      ],
       ...,
       [ 0.21      ,  3.9499998 ,  5.48      , ...,  0.02      ,
        -0.53999996,  0.7       ],
       [-2.7       ,  0.        ,  2.71      , ..., -2.87      ,
        -1.23      , -3.5       ],
       [ 0.22      ,  3.74      ,  4.94      , ..., -0.17999999,
         0.22      ,  0.44      ]], dtype=float32)
Coordinates:
    longitude  (city) float64 9.875 11.12 6.875 9.625 ... 7.375 9.125 7.125
    latitude   (city) float64 53.62 49.38 51.12 52.38 ... 51.62 48.88 51.38
  * time       (time) datetime64[ns] 1950-01-01 1950-01-02 ... 2025-12-31
  * city       (city) <U20 'Hamburg Hbf' 'Nürnberg Hbf' ... 'Essen Hbf'
Attributes:
    units:          Celsius
    long_name:      mean temperature
    standard_name:  air_temperature

In [ ]:
city_data.to_netcdf('city_temperature.nc')

### Get data with lat/lon

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import collections
import xarray as xr

In [2]:
city_locations = {'Hamburg Hbf': (53.55073, 9.99302),
 'Berlin Hbf': (52.52501, 13.36962),
 'Düsseldorf Hbf': (51.22172, 6.77616),
 'Hannover Hbf': (52.37052, 9.73322),
 'Köln Hbf': (50.93333, 6.95),
 'München Hbf': (48.06489, 11.66327),
 'Frankfurt Hbf': (50.11552, 8.68417),
 'Dortmund Hbf': (51.51494, 7.466),
 'Stuttgart Hbf': (48.78232, 9.17702),
 'Essen Hbf': (51.45657, 7.01228)}

In [3]:
variables = ['tn', 'tx', 'tg']
city_data = {}
for var in variables:
    with xr.open_dataset(f'/work/bb1445/b383514/other/train_disruption/eobs/{var}_ens_mean_0.25deg_reg_v33.0e.nc', chunks={'time': 365}) as nc:
        city_data[var] = nc[var].sel(latitude=lats, longitude=lons, method='nearest').load()

NameError: name 'lats' is not defined

In [4]:
import xarray as xr

variables = ['tn', 'tx', 'tg']
names = list(city_locations)

lats = xr.DataArray([city_locations[n][0] for n in names],
                    dims="city", coords={"city": names})
lons = xr.DataArray([city_locations[n][1] for n in names],
                    dims="city", coords={"city": names})

city_data = {}
for var in variables:
    with xr.open_dataset(
        f'/work/bb1445/b383514/other/train_disruption/eobs/{var}_ens_mean_0.25deg_reg_v33.0e.nc',
        chunks={'time': 365},
    ) as nc:
        city_data[var] = nc[var].sel(latitude=lats, longitude=lons,
                                     method='nearest').load()

In [5]:
city_data

{'tn': <xarray.DataArray 'tn' (time: 27759, city: 10)>
 array([[-3.12      , -4.45      , -3.35      , ..., -5.0099998 ,
         -5.35      , -3.56      ],
        [-0.64      , -3.55      , -1.66      , ..., -0.14      ,
         -5.02      , -0.55      ],
        [ 2.7       ,  1.4499999 ,  3.56      , ...,  4.19      ,
          0.57      ,  3.78      ],
        ...,
        [-0.98999995, -1.05      , -5.69      , ..., -4.17      ,
         -7.5499997 , -5.0899997 ],
        [-4.29      , -2.58      , -4.36      , ..., -4.12      ,
         -6.2599998 , -2.97      ],
        [-1.4499999 , -3.57      , -4.35      , ..., -3.86      ,
         -8.7699995 , -3.07      ]], dtype=float32)
 Coordinates:
     longitude  (city) float64 9.875 13.38 6.875 9.625 ... 7.375 9.125 7.125
     latitude   (city) float64 53.62 52.62 51.12 52.38 ... 51.62 48.88 51.38
   * time       (time) datetime64[ns] 1950-01-01 1950-01-02 ... 2025-12-31
   * city       (city) <U14 'Hamburg Hbf' 'Berlin Hbf' ... 'E

In [7]:
ds_cities = xr.Dataset(city_data)

In [8]:
ds_cities.to_netcdf('/work/bb1445/b383514/other/train_disruption/eobs/city_tg_tn_tx.nc')

In [9]:
from datasets import load_dataset

ds = load_dataset("piebro/deutsche-bahn-data")

ModuleNotFoundError: No module named 'datasets'